# Forecast Evaluation and Professional Benchmarking

**Portfolio version.** This notebook scores market-implied CPI forecasts against realized BLS inflation and compares directional performance with the Cleveland Fed inflation nowcast and naive baselines. The BLS API key is read from an environment variable and is not stored in the repository.


# Forecast Evaluation: BLS Outcomes, Brier Scores, and Benchmarks 

Pulls the CPI-U index (series `CUUR0000SA0`: all items, U.S. city average) from the BLS public API, computes the 12-month percent change
the Kalshi YoY contracts settle on — and produces:

1. a date-matched calendar joining each contract to its realized value,
2. Brier score per release/horizon (from the per-threshold probabilities), and
3. a confidence-weighted directional test (from the point forecasts).


## Configuration

In [3]:
import os, json, urllib.request
from decimal import Decimal, ROUND_HALF_UP

import numpy as np
import pandas as pd

IN_DIR  = os.path.join('data', 'processed')
OUT_DIR = 'results'
os.makedirs(OUT_DIR, exist_ok=True)

BLS_URL    = 'https://api.bls.gov/publicAPI/v2/timeseries/data/'
CPI_SERIES = 'CUUR0000SA0'                    # CPI-U, all items, U.S. city average, NSA
API_KEY    = os.getenv('BLS_API_KEY')  # optional; BLS public API also permits unauthenticated requests

MONTHS = {'JAN':1,'FEB':2,'MAR':3,'APR':4,'MAY':5,'JUN':6,
          'JUL':7,'AUG':8,'SEP':9,'OCT':10,'NOV':11,'DEC':12}
HORIZONS = ['21d','14d','7d','3d','1d','final']

## Release dates (annotation only)

Release dates are not carried in the BLS timeseries API, so they're kept separately here. They
label the calendar output only and enter **none** of the scores — you can empty this dict and the
Brier/directional numbers are unchanged. The 2025 entries reflect the shutdown reschedules
(Sep-25 → Oct 24, Nov-25 → Dec 18; Oct-25 was never published).

In [6]:
RELEASE_DATES = {
    '2022-11':'2022-12-13','2022-12':'2023-01-12','2023-01':'2023-02-14',
    '2023-02':'2023-03-14','2023-03':'2023-04-12','2023-04':'2023-05-10',
    '2023-05':'2023-06-13','2023-06':'2023-07-12','2023-07':'2023-08-10',
    '2023-08':'2023-09-13','2023-09':'2023-10-12','2023-10':'2023-11-14',
    '2023-11':'2023-12-12','2023-12':'2024-01-11','2024-01':'2024-02-13',
    '2024-02':'2024-03-12','2024-03':'2024-04-10','2024-04':'2024-05-15',
    '2024-05':'2024-06-12','2024-06':'2024-07-11','2024-07':'2024-08-14',
    '2024-08':'2024-09-11','2024-09':'2024-10-10','2024-10':'2024-11-13',
    '2024-11':'2024-12-11','2024-12':'2025-01-15','2025-01':'2025-02-12',
    '2025-02':'2025-03-12','2025-03':'2025-04-10','2025-04':'2025-05-13',
    '2025-05':'2025-06-11','2025-06':'2025-07-15','2025-07':'2025-08-12',
    '2025-08':'2025-09-11','2025-09':'2025-10-24','2025-11':'2025-12-18',
    '2025-12':'2026-01-13','2026-01':'2026-02-13','2026-02':'2026-03-11',
    '2026-03':'2026-04-10','2026-04':'2026-05-12','2026-05':'2026-06-10',
    '2026-06':'2026-07-14','2026-07':'2026-08-12',
}

## Pull the CPI index and compute year-over-year

`fetch_cpi_index` POSTs to the BLS API and returns index levels keyed by month; `yoy_from_index`
turns those into the 12-month percent change with half-up rounding to one decimal, matching BLS
headline rounding. A month that wasn't published (e.g. Oct 2025) is simply absent, and the YoY
step skips over it.

In [9]:
def round1(x):
    """Half-up rounding to one decimal, to match BLS headline rounding."""
    return float(Decimal(str(x)).quantize(Decimal('0.1'), rounding=ROUND_HALF_UP))


def fetch_cpi_index(start_year, end_year, series=CPI_SERIES, key=API_KEY):
    """Return {'YYYY-MM': index_level} from the BLS API for the given span."""
    payload = {'seriesid': [series], 'startyear': str(start_year), 'endyear': str(end_year)}
    if key:
        payload['registrationkey'] = key

    req = urllib.request.Request(BLS_URL, data=json.dumps(payload).encode(),
                                 headers={'Content-Type': 'application/json'})
    with urllib.request.urlopen(req, timeout=30) as resp:
        body = json.load(resp)

    if body.get('status') != 'REQUEST_SUCCEEDED':
        raise RuntimeError(f"BLS API: {body.get('status')} {body.get('message')}")

    index = {}
    for point in body['Results']['series'][0]['data']:
        if point['period'] == 'M13':          # annual average, skip
            continue
        raw = point['value'].replace(',', '').strip()
        try:
            value = float(raw)
        except ValueError:
            continue                          # '-' or other placeholder -> month is missing
        index[f"{int(point['year']):04d}-{int(point['period'][1:]):02d}"] = value
    return index

def yoy_from_index(index):
    """12-month percent change per month, gap-aware."""
    out = {}
    for ym, level in index.items():
        y, m = map(int, ym.split('-'))
        prior = index.get(f'{y-1:04d}-{m:02d}')
        if prior:
            out[ym] = round1((level / prior - 1) * 100)
    return out

## Join contracts to realized values

Each contract maps to its **reference month** (the month inflation is measured, released the
following month). The directional baseline is the prior release; where a month is missing the
baseline walks back to the last published reading.

In [12]:
def ref_month(ticker):
    """CPIYOY-22DEC / KXCPIYOY-26JUN -> '2022-12' / '2026-06'."""
    tail = ticker.split('-')[-1]
    return f'{2000 + int(tail[:2]):04d}-{MONTHS[tail[2:]]:02d}'


def prev_month(ym):
    y, m = map(int, ym.split('-'))
    return f'{y-1:04d}-12' if m == 1 else f'{y:04d}-{m-1:02d}'


def build_calendar(tickers, yoy):
    rows = []
    for t in sorted(tickers):
        ym = ref_month(t)
        actual = yoy.get(ym)

        base_m, base_v, note, steps = prev_month(ym), None, '', 0
        while base_v is None and steps < 3:          # walk back over any gaps
            base_v = yoy.get(base_m)
            if base_v is None:
                base_m = prev_month(base_m); steps += 1
        if steps:
            note = 'prior month missing; baseline = last published reading'
        if actual is None:
            note = (note + '; ' if note else '') + 'no actual (CPI not published)'

        change = None if actual is None or base_v is None else round1(actual - base_v)
        direction = None if change is None else (
            'up' if change > 0.05 else 'down' if change < -0.05 else 'flat')

        rows.append([t, ym, RELEASE_DATES.get(ym), actual, base_m, base_v,
                     change, direction, note])

    return pd.DataFrame(rows, columns=[
        'event_ticker','ref_month','release_date','actual_yoy','baseline_month',
        'baseline_yoy','realized_change','realized_direction','notes'])

## Brier score per release / horizon

Each row of the adjusted panel is a `CPI YoY >= threshold` contract; the outcome is 1 when the
realized value clears the strike (a strike exactly equal to the actual counts as a hit).

In [15]:
def brier_scores(adjusted, calendar):
    actual = calendar.set_index('event_ticker')['actual_yoy']
    df = adjusted.copy()
    df['actual'] = df['event_ticker'].map(actual)
    df = df.dropna(subset=['actual'])

    df['outcome'] = (df['actual'] >= df['threshold']).astype(int)
    df['sq_err'] = (df['S_adjusted'] - df['outcome']) ** 2

    return (df.groupby(['event_ticker','horizon'])
              .agg(n_thresholds=('sq_err','size'), brier=('sq_err','mean'))
              .reset_index())

## Confidence-weighted directional test

Does the point forecast call the direction versus the prior release, weighted by the size of the
predicted move (a proxy for the market's conviction)? Events with no actual/baseline/forecast or a
flat realized move are dropped — there's no direction to score.

In [18]:
def directional_test(summary, calendar, point='median'):
    df = summary.merge(
        calendar[['event_ticker','baseline_yoy','actual_yoy','realized_direction']],
        on='event_ticker', how='left')
    df = df.dropna(subset=[point, 'baseline_yoy', 'realized_direction'])
    df = df[df['realized_direction'] != 'flat'].copy()

    move = df[point] - df['baseline_yoy']
    df['pred_direction'] = np.where(move > 0, 'up', np.where(move < 0, 'down', 'flat'))
    df['correct'] = (df['pred_direction'] == df['realized_direction']).astype(int)
    df['weight'] = move.abs()

    def weighted(g):
        return np.average(g['correct'], weights=g['weight']) if g['weight'].sum() else np.nan

    by_h = (df.groupby('horizon')
              .apply(lambda g: pd.Series({
                  'n_releases': len(g),
                  'hit_rate': g['correct'].mean(),
                  'confidence_wtd_hit_rate': weighted(g),
              }), include_groups=False)
              .reindex(HORIZONS).reset_index())
    return by_h, df

## Run

In [21]:
adjusted = pd.read_csv(os.path.join(IN_DIR, 'forecast_panel_adjusted.csv'))
summary  = pd.read_csv(os.path.join(IN_DIR, 'forecast_panel_summary.csv'))

tickers   = set(adjusted.event_ticker) | set(summary.event_ticker)
ref_years = [int(ref_month(t)[:4]) for t in tickers]
index = fetch_cpi_index(min(ref_years) - 1, max(ref_years))   # -1 for the YoY base
yoy   = yoy_from_index(index)

calendar = build_calendar(tickers, yoy)
brier = brier_scores(adjusted, calendar)
direction_by_h, direction_rows = directional_test(summary, calendar, 'median')

print(f'Pulled {len(yoy)} monthly YoY values from BLS ({CPI_SERIES}).')
print(f'Scored {brier.event_ticker.nunique()} releases '
      f'({int(calendar.actual_yoy.isna().sum())} excluded: not published).')

Pulled 54 monthly YoY values from BLS (CUUR0000SA0).
Scored 43 releases (1 excluded: not published).


### Brier by horizon  
_naive 0.5 forecast = 0.25_

In [23]:
brier.groupby('horizon')['brier'].mean().reindex(HORIZONS).round(4).to_frame('mean_brier')

,mean_brier
horizon,
21d,0.0854
14d,0.0718
7d,0.0705
3d,0.0648
1d,0.0659
final,0.0659


### Directional accuracy by horizon (median forecast)

In [27]:
pooled_u = direction_rows.correct.mean()
pooled_w = np.average(direction_rows.correct, weights=direction_rows.weight)
print(f'Pooled: {pooled_u:.1%} unweighted, {pooled_w:.1%} confidence-weighted ({len(direction_rows)} calls)')
direction_by_h.round(4)

Pooled: 97.4% unweighted, 99.9% confidence-weighted (230 calls)


,horizon,n_releases,hit_rate,confidence_wtd_hit_rate
0,21d,37.0,0.9459,0.9950
1,14d,38.0,0.9737,0.9992
2,7d,38.0,0.9737,0.9999
3,3d,39.0,1.0000,1.0000
4,1d,39.0,0.9744,0.9997
5,final,39.0,0.9744,0.9997


### Save outputs

In [30]:
calendar.to_csv(f'{OUT_DIR}/cpi_actuals_calendar.csv', index=False)
brier.to_csv(f'{OUT_DIR}/brier_by_release_horizon.csv', index=False)
direction_by_h.to_csv(f'{OUT_DIR}/directional_by_horizon_median.csv', index=False)
detail_cols = ['event_ticker','baseline_yoy','median','actual_yoy',
               'pred_direction','realized_direction','correct','weight']
(direction_rows[direction_rows.horizon == '1d'][detail_cols]
 .to_csv(f'{OUT_DIR}/directional_detail_1d.csv', index=False))
print('wrote cpi_actuals_calendar.csv, brier_by_release_horizon.csv,')
print('      directional_by_horizon_median.csv, directional_detail_1d.csv')

wrote cpi_actuals_calendar.csv, brier_by_release_horizon.csv,
      directional_by_horizon_median.csv, directional_detail_1d.csv


In [32]:
import os
from datetime import date, timedelta

DATA_DIR = 'cleveland_nowcast'  # optional local folder containing Cleveland Fed vintage CSVs
                       # Professional-benchmark source files are not redistributed in this repository.

HZ_DAYS = {'21d': 21, '14d': 14, '7d': 7, '3d': 3, '1d': 1}   # days before release

# --- read every nowcast file by constructed name, Nov-2022 .. Jun-2026 -------
def month_range(start=(2022, 10), end=(2026, 6)):
    y, m = start
    while (y, m) <= end:
        yield y, m
        m += 1
        if m == 13:
            y, m = y + 1, 1

nowcast_daily = {}
missing = []
for ty, tm in month_range():
    fname = f'Year-Over-YearPercentChange-{ty}-{tm}.csv'      # e.g. 2024-7, 2026-6 (no zero-pad)
    path = os.path.join(DATA_DIR, fname)
    if not os.path.exists(path):
        missing.append(fname)
        continue
    df = pd.read_csv(path)
    daily, year, prev_mnum = {}, ty, tm
    for _, row in df.iterrows():
        mm, dd = map(int, str(row['Label']).split('/'))
        if mm < prev_mnum:                 # month rolled over -> next calendar year
            year += 1
        prev_mnum = mm
        val = row['CPI Inflation']         # headline NSA YoY nowcast
        if pd.notna(val) and str(val).strip() != '':
            daily[date(year, mm, dd)] = float(val)
    nowcast_daily[f'{ty:04d}-{tm:02d}'] = daily

print(f'read {len(nowcast_daily)} monthly files:',
      min(nowcast_daily), '->', max(nowcast_daily))
if missing:
    print('not found (ok to ignore 2025-10):', missing)

# --- sample the nowcast at each horizon, per reference month -----------------
def _asof(daily, target, max_back=10):
    d = target
    for _ in range(max_back + 1):
        if d in daily:
            return daily[d]
        d -= timedelta(days=1)
    return None

cal = calendar.dropna(subset=['actual_yoy']).set_index('ref_month')

nc_rows = []
for ym, daily in nowcast_daily.items():
    if ym not in cal.index or not daily:
        continue
    R = date.fromisoformat(cal.loc[ym, 'release_date'])
    for h, k in HZ_DAYS.items():
        v = _asof(daily, R - timedelta(days=k))
        if v is not None:
            nc_rows.append({'ref_month': ym, 'horizon': h, 'nowcast_yoy': v})
    nc_rows.append({'ref_month': ym, 'horizon': 'final', 'nowcast_yoy': daily[max(daily)]})
nowcast_panel = pd.DataFrame(nc_rows)

read 45 monthly files: 2022-10 -> 2026-06


In [34]:
def _dir(x, tol=0.0):
    return 'up' if x > tol else 'down' if x < -tol else 'flat'

# momentum: sign of the last observed month-over-month change in YoY, gap-aware
def _momentum_call(ym, max_steps=6):
    def back(m):                       # last published YoY strictly before month m
        for _ in range(max_steps):
            m = prev_month(m); v = yoy.get(m)
            if v is not None: return m, v
        return None, None
    m1, v1 = back(ym)
    if v1 is None: return None
    _, v2 = back(m1)
    if v2 is None: return None
    return _dir(v1 - v2)

# realized direction + baseline come from `calendar`; keep non-flat scored months
base = calendar.dropna(subset=['actual_yoy', 'baseline_yoy', 'realized_direction'])
base = base[base['realized_direction'] != 'flat'][
    ['ref_month', 'baseline_yoy', 'realized_direction']].set_index('ref_month')
majority_dir = base['realized_direction'].value_counts().idxmax()

# our forecast, sampled at each horizon from `summary` (median point forecast)
ours = summary.merge(calendar[['event_ticker', 'ref_month']], on='event_ticker')
ours = ours.merge(base, left_on='ref_month', right_index=True).dropna(subset=['median'])
ours['pred'] = [_dir(m - b) for m, b in zip(ours['median'], ours['baseline_yoy'])]
ours['correct'] = (ours['pred'] == ours['realized_direction']).astype(int)
ours['weight'] = (ours['median'] - ours['baseline_yoy']).abs()

# nowcast, sampled at each horizon from `nowcast_panel`
nc = nowcast_panel.merge(base, left_on='ref_month', right_index=True)
nc['pred'] = [_dir(v - b) for v, b in zip(nc['nowcast_yoy'], nc['baseline_yoy'])]
nc['correct'] = (nc['pred'] == nc['realized_direction']).astype(int)
nc['weight'] = (nc['nowcast_yoy'] - nc['baseline_yoy']).abs()

def _score(df, months, weighted=True):
    d = df[df['ref_month'].isin(months)]
    if len(d) == 0:
        return (np.nan, np.nan, 0)
    hit = d['correct'].mean()
    wtd = (np.average(d['correct'], weights=d['weight'])
           if weighted and d['weight'].sum() else np.nan)
    return (hit, wtd, len(d))

rows = []
for h in HORIZONS:
    o_h = ours[ours.horizon == h]
    n_h = nc[nc.horizon == h]
    common = sorted(set(o_h.ref_month) & set(n_h.ref_month))   # identical months per method
    if not common:
        continue
    oh, ow, n = _score(o_h, common)
    nh, nw, _ = _score(n_h, common)
    mom_pairs = [(_momentum_call(m), base.loc[m, 'realized_direction']) for m in common]
    mom = np.mean([c == r for c, r in mom_pairs if c is not None])
    maj = np.mean([majority_dir == base.loc[m, 'realized_direction'] for m in common])
    rows.append({'horizon': h, 'n_months': n,
                 'ours': round(oh, 3), 'ours_wtd': round(ow, 3),
                 'nowcast': round(nh, 3), 'nowcast_wtd': round(nw, 3),
                 'momentum': round(mom, 3), 'majority': round(maj, 3)})

benchmark = pd.DataFrame(rows).set_index('horizon').reindex(HORIZONS).dropna(how='all')
print('Directional hit rate by horizon (same months across methods):')
print(benchmark.to_string())

# point-forecast error (MAE, pts) at the 1d horizon: ours vs nowcast vs persistence
o1 = ours[ours.horizon == '1d'].merge(calendar[['ref_month', 'actual_yoy']], on='ref_month')
n1 = nc[nc.horizon == '1d'].merge(calendar[['ref_month', 'actual_yoy']], on='ref_month')
mae_ours = (o1['median'] - o1['actual_yoy']).abs().mean()
mae_now  = (n1['nowcast_yoy'] - n1['actual_yoy']).abs().mean()
mae_pers = np.mean([abs(cal.loc[m, 'actual_yoy'] - base.loc[m, 'baseline_yoy'])
                    for m in base.index if m in cal.index])
print(f'\nPoint-forecast MAE @1d (pts): ours {mae_ours:.3f} | '
      f'nowcast {mae_now:.3f} | persistence {mae_pers:.3f}')

benchmark.to_csv(f'{OUT_DIR}/benchmark_directional.csv')

Directional hit rate by horizon (same months across methods):
         n_months   ours  ours_wtd  nowcast  nowcast_wtd  momentum  majority
horizon                                                                     
21d            37  0.946     0.995    0.892        0.971     0.622     0.568
14d            38  0.974     0.999    0.921        0.976     0.632     0.579
7d             38  0.974     1.000    0.921        0.976     0.632     0.579
3d             39  1.000     1.000    0.923        0.976     0.641     0.590
1d             39  0.974     1.000    0.923        0.976     0.641     0.590
final          39  0.974     1.000    0.923        0.976     0.641     0.590

Point-forecast MAE @1d (pts): ours 0.075 | nowcast 0.122 | persistence 0.338
